# Auto Scene Assessment Agent
### ITAI 1378 – Computer Vision Final Project
**Student:** Courtney Bernard  
**Project Tier:** Tier 2

## Project Overview
This project builds a Tier 2 computer vision agent that analyzes vehicle and roadway images using two computer vision tools: YOLO and CLIP.

YOLO is used to detect objects such as cars, trucks, motorcycles, people, and traffic-related objects. CLIP is used to analyze the overall context of the image. The agent combines the results from both models to make a rule-based decision about the scene and produce a structured assessment.

The goal is not to replace human judgment. The agent is designed to demonstrate how computer vision tools can work together to assist with reviewing vehicle and roadway scenes.

## Agent Pipeline
**Input → Preprocessing → YOLO Detection + CLIP Classification → Rule-Based Reasoning → Action/Output → Logging**

In [6]:
!pip install -q ultralytics transformers torch torchvision pillow

In [7]:
import os
import json
import time
from pathlib import Path

import torch
from PIL import Image
import matplotlib.pyplot as plt

from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Imports successful!")
print("Device:", device)

Imports successful!
Device: cuda


In [8]:
# Tool 1: YOLO object detection
yolo_model = YOLO("yolov8n.pt")

# Tool 2: CLIP image classification
clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

print("YOLO loaded successfully!")
print("CLIP loaded successfully!")
print("Tier 2 vision tools are ready.")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

YOLO loaded successfully!
CLIP loaded successfully!
Tier 2 vision tools are ready.


## Step 2: Input Ingestion

The agent needs to accept real-world image inputs before it can analyze them. Instead of using one hard-coded image, this section creates the project folders and allows multiple images to be uploaded for testing.

The uploaded images will later move through preprocessing, YOLO object detection, CLIP image understanding, decision-making, and output generation.

In [9]:
# Cell 4 - Create project folders

import os

folders = [
    "data/sample",
    "results/images",
    "results/traces",
    "models",
    "docs"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully!")

for folder in folders:
    print(f"✓ {folder}")

Project folders created successfully!
✓ data/sample
✓ results/images
✓ results/traces
✓ models
✓ docs


In [10]:
# Cell 5 - Upload multiple test images

from google.colab import files
import os
import shutil

uploaded = files.upload()

sample_folder = "data/sample"

for filename in uploaded.keys():
    destination = os.path.join(sample_folder, filename)

    if os.path.exists(filename):
        shutil.move(filename, destination)

print("\nUploaded test images:")

for filename in os.listdir(sample_folder):
    print(f"✓ {filename}")

Saving sunset.jpg to sunset.jpg

Uploaded test images:
✓ sunset.jpg


In [12]:
# Cell 5B - Add additional test images

from google.colab import files
import os
import shutil

uploaded = files.upload()

sample_folder = "data/sample"

for filename in uploaded.keys():
    destination = os.path.join(sample_folder, filename)

    if os.path.exists(filename):
        shutil.move(filename, destination)

print("\nCurrent test images:")

for filename in os.listdir(sample_folder):
    print(f"✓ {filename}")

Saving morocco market.avif to morocco market.avif
Saving airport.jpg to airport.jpg
Saving car lot.jpg to car lot.jpg
Saving dogs.jpg to dogs.jpg

Current test images:
✓ sunset.jpg
✓ airport.jpg
✓ car lot.jpg
✓ dogs.jpg
✓ morocco market.avif


## Step 3: Image Validation and Preprocessing

Before the computer vision models analyze an image, the agent checks that each input is a valid image and prepares it for processing.

This step helps prevent bad or corrupted files from crashing the system. Valid images are converted to RGB format and their basic information is recorded before they are sent to the vision models.

In [13]:
# Cell 6 - Validate and preprocess input images

from PIL import Image
import os

sample_folder = "data/sample"

valid_images = []
invalid_images = []

supported_extensions = (".jpg", ".jpeg", ".png", ".webp", ".avif")

print("Validating input images...\n")

for filename in os.listdir(sample_folder):
    file_path = os.path.join(sample_folder, filename)

    try:
        # Check file extension
        if not filename.lower().endswith(supported_extensions):
            raise ValueError("Unsupported image format")

        # Open and verify image
        with Image.open(file_path) as img:
            img.verify()

        # Reopen after verify and convert to RGB
        with Image.open(file_path) as img:
            rgb_image = img.convert("RGB")
            width, height = rgb_image.size

        valid_images.append(file_path)

        print(f"✓ VALID: {filename}")
        print(f"  Size: {width} x {height}")
        print(f"  Format: {filename.split('.')[-1].upper()}\n")

    except Exception as e:
        invalid_images.append({
            "file": filename,
            "error": str(e)
        })

        print(f"✗ INVALID: {filename}")
        print(f"  Reason: {e}\n")

print("-" * 40)
print(f"Valid images: {len(valid_images)}")
print(f"Invalid images: {len(invalid_images)}")

Validating input images...

✓ VALID: sunset.jpg
  Size: 352 x 220
  Format: JPG

✓ VALID: airport.jpg
  Size: 300 x 220
  Format: JPG

✓ VALID: car lot.jpg
  Size: 386 x 220
  Format: JPG

✓ VALID: dogs.jpg
  Size: 392 x 220
  Format: JPG

✓ VALID: morocco market.avif
  Size: 1380 x 773
  Format: AVIF

----------------------------------------
Valid images: 5
Invalid images: 0


## Step 4: Object Detection with YOLO

The first computer vision tool used by the agent is YOLO. YOLO scans each valid image and identifies recognizable objects such as people, cars, animals, furniture, and other common items.

Instead of only printing the raw model output, the detections are converted into structured results containing the object label, confidence score, and bounding box coordinates. These structured results can later be used by the agent's decision-making logic.

In [14]:
# Cell 7 - Run YOLO object detection on valid images

import os
import json
import time

yolo_results = {}

print("Running YOLO object detection...\n")

for image_path in valid_images:
    filename = os.path.basename(image_path)

    start_time = time.time()

    results = yolo_model(image_path, verbose=False)

    inference_time = time.time() - start_time

    detections = []

    for result in results:
        if result.boxes is not None:
            for box in result.boxes:
                class_id = int(box.cls[0])
                confidence = float(box.conf[0])
                coordinates = box.xyxy[0].tolist()

                detection = {
                    "label": yolo_model.names[class_id],
                    "confidence": round(confidence, 3),
                    "bbox": [round(value, 2) for value in coordinates]
                }

                detections.append(detection)

    yolo_results[filename] = {
        "detections": detections,
        "object_count": len(detections),
        "inference_time_seconds": round(inference_time, 3)
    }

    print(f"IMAGE: {filename}")
    print(f"Objects detected: {len(detections)}")

    if detections:
        for detection in detections:
            print(
                f"  ✓ {detection['label']} "
                f"({detection['confidence']:.1%} confidence)"
            )
    else:
        print("  No objects detected.")

    print(f"Inference time: {inference_time:.3f} seconds")
    print("-" * 40)


Running YOLO object detection...

IMAGE: sunset.jpg
Objects detected: 0
  No objects detected.
Inference time: 1.480 seconds
----------------------------------------
IMAGE: airport.jpg
Objects detected: 12
  ✓ person (81.5% confidence)
  ✓ person (74.4% confidence)
  ✓ person (72.7% confidence)
  ✓ person (70.8% confidence)
  ✓ person (67.8% confidence)
  ✓ person (66.0% confidence)
  ✓ person (62.9% confidence)
  ✓ person (51.5% confidence)
  ✓ backpack (50.7% confidence)
  ✓ person (38.1% confidence)
  ✓ person (28.7% confidence)
  ✓ person (27.1% confidence)
Inference time: 0.090 seconds
----------------------------------------
IMAGE: car lot.jpg
Objects detected: 17
  ✓ car (75.3% confidence)
  ✓ car (64.7% confidence)
  ✓ car (62.3% confidence)
  ✓ car (50.0% confidence)
  ✓ car (49.2% confidence)
  ✓ car (48.4% confidence)
  ✓ car (48.2% confidence)
  ✓ car (47.1% confidence)
  ✓ car (46.4% confidence)
  ✓ car (44.0% confidence)
  ✓ car (38.4% confidence)
  ✓ bus (37.6% confidenc

## Step 5: Save Detection Results

The YOLO detection results are saved so the agent has a permanent record of what it detected in each image.

Structured detection data is stored as JSON, and annotated images with bounding boxes and labels are saved in the results folder. These files will later be used for evaluation, failure analysis, and the final GitHub repository.

In [15]:
# Cell 8 - Save YOLO detection results and annotated images

import os
import json

results_folder = "results/images"
os.makedirs(results_folder, exist_ok=True)

# Save structured YOLO results
with open("results/yolo_results.json", "w") as f:
    json.dump(yolo_results, f, indent=4)

print("✓ Saved structured results to results/yolo_results.json\n")

# Save annotated versions of each image
for image_path in valid_images:
    filename = os.path.basename(image_path)

    results = yolo_model(image_path, verbose=False)

    annotated_image = results[0].plot()

    output_name = f"annotated_{os.path.splitext(filename)[0]}.jpg"
    output_path = os.path.join(results_folder, output_name)

    from PIL import Image
    Image.fromarray(annotated_image[..., ::-1]).save(output_path)

    print(f"✓ Saved: {output_path}")

print("\nYOLO outputs saved successfully!")

✓ Saved structured results to results/yolo_results.json

✓ Saved: results/images/annotated_sunset.jpg
✓ Saved: results/images/annotated_airport.jpg
✓ Saved: results/images/annotated_car lot.jpg
✓ Saved: results/images/annotated_dogs.jpg
✓ Saved: results/images/annotated_morocco market.jpg

YOLO outputs saved successfully!


In [16]:
# Cell 9 - Load CLIP for scene understanding

from transformers import CLIPProcessor, CLIPModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading CLIP on {device}...")

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
).to(device)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

print("✓ CLIP loaded successfully!")
print("✓ Second computer vision tool ready!")

Loading CLIP on cuda...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

✓ CLIP loaded successfully!
✓ Second computer vision tool ready!


In [17]:
# Cell 10 - Use CLIP to classify the overall scene in each image

scene_labels = [
    "an airport scene",
    "a parking lot with cars",
    "a dog scene",
    "a busy outdoor market",
    "a sunset landscape",
    "a road or traffic scene"
]

clip_results = []

for image_path in valid_images:
    image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(
        text=scene_labels,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = clip_model(**inputs)

    probabilities = outputs.logits_per_image.softmax(dim=1)[0]
    best_index = probabilities.argmax().item()

    result = {
        "image": os.path.basename(image_path),
        "scene": scene_labels[best_index],
        "confidence": round(probabilities[best_index].item() * 100, 1)
    }

    clip_results.append(result)

    print(f"IMAGE: {result['image']}")
    print(f"CLIP scene: {result['scene']}")
    print(f"Confidence: {result['confidence']}%")
    print("-" * 40)

print("\n✓ CLIP scene analysis complete!")

IMAGE: sunset.jpg
CLIP scene: a sunset landscape
Confidence: 99.9%
----------------------------------------
IMAGE: airport.jpg
CLIP scene: an airport scene
Confidence: 100.0%
----------------------------------------
IMAGE: car lot.jpg
CLIP scene: a parking lot with cars
Confidence: 97.3%
----------------------------------------
IMAGE: dogs.jpg
CLIP scene: a dog scene
Confidence: 99.1%
----------------------------------------
IMAGE: morocco market.avif
CLIP scene: a busy outdoor market
Confidence: 99.3%
----------------------------------------

✓ CLIP scene analysis complete!


In [18]:
# Cell 11 - Save CLIP scene classification results

with open("results/clip_results.json", "w") as f:
    json.dump(clip_results, f, indent=4)

print("✓ Saved structured CLIP results to results/clip_results.json")

✓ Saved structured CLIP results to results/clip_results.json


In [21]:
# Cell 12 - Agent reasoning: combine YOLO and CLIP results

agent_results = []

for clip_result in clip_results:
    image_name = clip_result["image"]

    # Get matching YOLO result using the image filename
    matching_yolo = yolo_results.get(image_name)

    if matching_yolo is None:
        print(f"⚠ No YOLO result found for {image_name}")
        continue

    detected_objects = [
        detection["label"]
        for detection in matching_yolo["detections"]
    ]

    scene = clip_result["scene"]
    confidence = clip_result["confidence"]

    # Rule-based reasoning
    if scene == "an airport scene":
        decision = "TRAVEL SCENE DETECTED"
        reason = "CLIP identified an airport scene and YOLO detected people and travel-related objects."

    elif scene == "a parking lot with cars":
        decision = "VEHICLE AREA DETECTED"
        reason = "CLIP identified a parking lot and YOLO detected multiple vehicles."

    elif scene == "a dog scene":
        decision = "ANIMAL SCENE DETECTED"
        reason = "CLIP identified a dog scene and YOLO detected animals."

    elif scene == "a busy outdoor market":
        decision = "CROWDED PUBLIC AREA DETECTED"
        reason = "CLIP identified a busy market scene and YOLO detected people and market-related objects."

    elif scene == "a sunset landscape":
        decision = "LOW-ACTIVITY SCENE"
        reason = "CLIP identified a sunset landscape and YOLO detected no objects."

    else:
        decision = "GENERAL SCENE"
        reason = "The image did not match one of the main scene categories."

    result = {
        "image": image_name,
        "scene": scene,
        "clip_confidence": confidence,
        "detected_objects": detected_objects,
        "object_count": matching_yolo["object_count"],
        "yolo_inference_time": matching_yolo["inference_time_seconds"],
        "decision": decision,
        "reason": reason
    }

    agent_results.append(result)

    print(f"IMAGE: {image_name}")
    print(f"Scene: {scene}")
    print(f"Objects: {detected_objects}")
    print(f"Decision: {decision}")
    print(f"Reason: {reason}")
    print("-" * 50)

print("\n✓ Agent reasoning complete!")


IMAGE: sunset.jpg
Scene: a sunset landscape
Objects: []
Decision: LOW-ACTIVITY SCENE
Reason: CLIP identified a sunset landscape and YOLO detected no objects.
--------------------------------------------------
IMAGE: airport.jpg
Scene: an airport scene
Objects: ['person', 'person', 'person', 'person', 'person', 'person', 'person', 'person', 'backpack', 'person', 'person', 'person']
Decision: TRAVEL SCENE DETECTED
Reason: CLIP identified an airport scene and YOLO detected people and travel-related objects.
--------------------------------------------------
IMAGE: car lot.jpg
Scene: a parking lot with cars
Objects: ['car', 'car', 'car', 'car', 'car', 'car', 'car', 'car', 'car', 'car', 'car', 'bus', 'car', 'car', 'boat', 'car', 'bus']
Decision: VEHICLE AREA DETECTED
Reason: CLIP identified a parking lot and YOLO detected multiple vehicles.
--------------------------------------------------
IMAGE: dogs.jpg
Scene: a dog scene
Objects: ['dog', 'bear', 'dog', 'dog']
Decision: ANIMAL SCENE DETE

In [22]:
# Cell 13 - Save combined agent results

with open("results/agent_results.json", "w") as f:
    json.dump(agent_results, f, indent=4)

print("✓ Saved agent decisions to results/agent_results.json")

✓ Saved agent decisions to results/agent_results.json


In [23]:
# Cell 14 - Save a trace log for each agent run

traces_folder = "results/traces"
os.makedirs(traces_folder, exist_ok=True)

for result in agent_results:
    image_name = result["image"]

    trace = {
        "input": image_name,
        "perception": {
            "yolo_objects": result["detected_objects"],
            "clip_scene": result["scene"],
            "clip_confidence": result["clip_confidence"]
        },
        "decision": {
            "result": result["decision"],
            "reason": result["reason"]
        },
        "action": {
            "saved_annotated_image": f"results/images/annotated_{os.path.splitext(image_name)[0]}.jpg",
            "saved_structured_result": "results/agent_results.json"
        }
    }

    trace_name = f"{os.path.splitext(image_name)[0]}_trace.json"
    trace_path = os.path.join(traces_folder, trace_name)

    with open(trace_path, "w") as f:
        json.dump(trace, f, indent=4)

    print(f"✓ Saved trace: {trace_path}")

print("\n✓ All agent traces saved successfully!")

✓ Saved trace: results/traces/sunset_trace.json
✓ Saved trace: results/traces/airport_trace.json
✓ Saved trace: results/traces/car lot_trace.json
✓ Saved trace: results/traces/dogs_trace.json
✓ Saved trace: results/traces/morocco market_trace.json

✓ All agent traces saved successfully!


In [24]:
# Cell 15 - Basic evaluation metrics

total_images = len(agent_results)

successful_runs = sum(
    1 for result in agent_results
    if result["decision"] is not None
)

success_rate = (successful_runs / total_images) * 100 if total_images > 0 else 0

average_yolo_time = sum(
    result["yolo_inference_time"] for result in agent_results
) / total_images if total_images > 0 else 0

print("AGENT EVALUATION SUMMARY")
print("-" * 40)
print(f"Total test images: {total_images}")
print(f"Successful end-to-end runs: {successful_runs}")
print(f"Task success rate: {success_rate:.1f}%")
print(f"Average YOLO inference time: {average_yolo_time:.3f} seconds")
print("-" * 40)

AGENT EVALUATION SUMMARY
----------------------------------------
Total test images: 5
Successful end-to-end runs: 5
Task success rate: 100.0%
Average YOLO inference time: 0.350 seconds
----------------------------------------


In [25]:
# Cell 16 - Save evaluation metrics

metrics_text = f"""
AUTO SCENE AGENT - EVALUATION METRICS
-------------------------------------
Total test images: {total_images}
Successful end-to-end runs: {successful_runs}
Task success rate: {success_rate:.1f}%
Average YOLO inference time: {average_yolo_time:.3f} seconds

Current evaluation uses the first 5 test images.
Additional test scenarios will be added for the final evaluation.
"""

with open("results/metrics.txt", "w") as f:
    f.write(metrics_text)

print("✓ Saved evaluation metrics to results/metrics.txt")

✓ Saved evaluation metrics to results/metrics.txt


In [26]:
# Cell 17 - Upload 5 additional test images

from google.colab import files

new_uploaded = files.upload()

print(f"✓ Uploaded {len(new_uploaded)} additional test images.")

Saving car.jpg to car.jpg
Saving downtown.jpg to downtown.jpg
Saving cars at night.jpg to cars at night.jpg
Saving beautiful.jpg to beautiful.jpg
Saving rain.jpg to rain.jpg
✓ Uploaded 5 additional test images.


In [27]:
# Cell 18 - Run YOLO on the 5 additional test images

new_images = list(new_uploaded.keys())

for image_path in new_images:
    start_time = time.time()

    results = yolo_model(image_path, verbose=False)
    inference_time = round(time.time() - start_time, 3)

    detections = []

    for box in results[0].boxes:
        class_id = int(box.cls[0])
        label = yolo_model.names[class_id]
        confidence = round(float(box.conf[0]), 3)
        bbox = [round(float(x), 2) for x in box.xyxy[0].tolist()]

        detections.append({
            "label": label,
            "confidence": confidence,
            "bbox": bbox
        })

    # Add results to existing YOLO dictionary
    filename = os.path.basename(image_path)

    yolo_results[filename] = {
        "detections": detections,
        "object_count": len(detections),
        "inference_time_seconds": inference_time
    }

    # Save annotated image
    annotated_image = results[0].plot()

    output_name = f"annotated_{os.path.splitext(filename)[0]}.jpg"
    output_path = os.path.join("results/images", output_name)

    Image.fromarray(annotated_image[..., ::-1]).save(output_path)

    print(f"IMAGE: {filename}")
    print(f"Objects detected: {len(detections)}")

    for detection in detections:
        print(
            f"  ✓ {detection['label']} "
            f"({detection['confidence'] * 100:.1f}% confidence)"
        )

    print(f"✓ Saved: {output_path}")
    print("-" * 45)

# Update saved YOLO results file with all 10 images
with open("results/yolo_results.json", "w") as f:
    json.dump(yolo_results, f, indent=4)

print("\n✓ Additional YOLO analysis complete!")
print(f"✓ Total YOLO test images now: {len(yolo_results)}")

IMAGE: car.jpg
Objects detected: 1
  ✓ car (92.3% confidence)
✓ Saved: results/images/annotated_car.jpg
---------------------------------------------
IMAGE: downtown.jpg
Objects detected: 0
✓ Saved: results/images/annotated_downtown.jpg
---------------------------------------------
IMAGE: cars at night.jpg
Objects detected: 2
  ✓ wine glass (38.3% confidence)
  ✓ wine glass (29.0% confidence)
✓ Saved: results/images/annotated_cars at night.jpg
---------------------------------------------
IMAGE: beautiful.jpg
Objects detected: 2
  ✓ bench (42.1% confidence)
  ✓ potted plant (30.4% confidence)
✓ Saved: results/images/annotated_beautiful.jpg
---------------------------------------------
IMAGE: rain.jpg
Objects detected: 0
✓ Saved: results/images/annotated_rain.jpg
---------------------------------------------

✓ Additional YOLO analysis complete!
✓ Total YOLO test images now: 10


In [28]:
# Cell 19 - Run CLIP on the 5 additional test images

for image_path in new_images:
    image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(
        text=scene_labels,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = clip_model(**inputs)

    probabilities = outputs.logits_per_image.softmax(dim=1)[0]
    best_index = probabilities.argmax().item()

    result = {
        "image": os.path.basename(image_path),
        "scene": scene_labels[best_index],
        "confidence": round(probabilities[best_index].item() * 100, 1)
    }

    clip_results.append(result)

    print(f"IMAGE: {result['image']}")
    print(f"CLIP scene: {result['scene']}")
    print(f"Confidence: {result['confidence']}%")
    print("-" * 40)

# Save updated CLIP results for all 10 images
with open("results/clip_results.json", "w") as f:
    json.dump(clip_results, f, indent=4)

print("\n✓ Additional CLIP analysis complete!")
print(f"✓ Total CLIP test images now: {len(clip_results)}")


IMAGE: car.jpg
CLIP scene: a road or traffic scene
Confidence: 64.1%
----------------------------------------
IMAGE: downtown.jpg
CLIP scene: a road or traffic scene
Confidence: 34.7%
----------------------------------------
IMAGE: cars at night.jpg
CLIP scene: a road or traffic scene
Confidence: 81.6%
----------------------------------------
IMAGE: beautiful.jpg
CLIP scene: a dog scene
Confidence: 77.0%
----------------------------------------
IMAGE: rain.jpg
CLIP scene: an airport scene
Confidence: 45.8%
----------------------------------------

✓ Additional CLIP analysis complete!
✓ Total CLIP test images now: 10


In [29]:
# Cell 20 - Final agent reasoning for all 10 test images

agent_results = []

for clip_result in clip_results:
    image_name = clip_result["image"]
    matching_yolo = yolo_results.get(image_name)

    if matching_yolo is None:
        print(f"⚠ No YOLO result found for {image_name}")
        continue

    detected_objects = [
        detection["label"]
        for detection in matching_yolo["detections"]
    ]

    scene = clip_result["scene"]
    confidence = clip_result["confidence"]

    # Rule-based reasoning with uncertainty handling
    if confidence < 50:
        decision = "MANUAL REVIEW RECOMMENDED"
        reason = "CLIP confidence is low, so the scene classification may be unreliable."

    elif scene == "an airport scene":
        decision = "TRAVEL SCENE DETECTED"
        reason = "CLIP identified an airport scene and YOLO provided object-level detections."

    elif scene == "a parking lot with cars":
        decision = "VEHICLE AREA DETECTED"
        reason = "CLIP identified a parking lot and YOLO detected vehicles."

    elif scene == "a road or traffic scene":
        if any(obj in ["car", "truck", "bus", "motorcycle"] for obj in detected_objects):
            decision = "TRAFFIC SCENE DETECTED"
            reason = "CLIP identified a road or traffic scene and YOLO detected vehicle-related objects."
        else:
            decision = "POSSIBLE TRAFFIC SCENE - REVIEW"
            reason = "CLIP identified a traffic scene, but YOLO did not confirm expected vehicle objects."

    elif scene == "a dog scene":
        if "dog" in detected_objects:
            decision = "ANIMAL SCENE DETECTED"
            reason = "CLIP identified a dog scene and YOLO also detected dogs."
        else:
            decision = "MODEL DISAGREEMENT - REVIEW"
            reason = "CLIP identified a dog scene, but YOLO did not detect a dog."

    elif scene == "a busy outdoor market":
        decision = "CROWDED PUBLIC AREA DETECTED"
        reason = "CLIP identified a busy market scene and YOLO detected people or market-related objects."

    elif scene == "a sunset landscape":
        decision = "LOW-ACTIVITY SCENE"
        reason = "CLIP identified a sunset landscape and YOLO detected little or no object activity."

    else:
        decision = "GENERAL SCENE"
        reason = "The scene did not strongly match a defined agent rule."

    result = {
        "image": image_name,
        "scene": scene,
        "clip_confidence": confidence,
        "detected_objects": detected_objects,
        "object_count": matching_yolo["object_count"],
        "yolo_inference_time": matching_yolo["inference_time_seconds"],
        "decision": decision,
        "reason": reason
    }

    agent_results.append(result)

    print(f"IMAGE: {image_name}")
    print(f"Scene: {scene} ({confidence}%)")
    print(f"Objects: {detected_objects}")
    print(f"Decision: {decision}")
    print(f"Reason: {reason}")
    print("-" * 55)

print("\n✓ Final reasoning completed for all test images!")
print(f"✓ Total agent results: {len(agent_results)}")

IMAGE: sunset.jpg
Scene: a sunset landscape (99.9%)
Objects: []
Decision: LOW-ACTIVITY SCENE
Reason: CLIP identified a sunset landscape and YOLO detected little or no object activity.
-------------------------------------------------------
IMAGE: airport.jpg
Scene: an airport scene (100.0%)
Objects: ['person', 'person', 'person', 'person', 'person', 'person', 'person', 'person', 'backpack', 'person', 'person', 'person']
Decision: TRAVEL SCENE DETECTED
Reason: CLIP identified an airport scene and YOLO provided object-level detections.
-------------------------------------------------------
IMAGE: car lot.jpg
Scene: a parking lot with cars (97.3%)
Objects: ['car', 'car', 'car', 'car', 'car', 'car', 'car', 'car', 'car', 'car', 'car', 'bus', 'car', 'car', 'boat', 'car', 'bus']
Decision: VEHICLE AREA DETECTED
Reason: CLIP identified a parking lot and YOLO detected vehicles.
-------------------------------------------------------
IMAGE: dogs.jpg
Scene: a dog scene (99.1%)
Objects: ['dog', 'b

In [30]:
# Cell 21 - Save final agent results for all 10 test images

with open("results/agent_results.json", "w") as f:
    json.dump(agent_results, f, indent=4)

print("✓ Saved final agent results to results/agent_results.json")
print(f"✓ Total saved agent results: {len(agent_results)}")

✓ Saved final agent results to results/agent_results.json
✓ Total saved agent results: 10


In [31]:
# Cell 22 - Save updated traces for all 10 agent runs

traces_folder = "results/traces"
os.makedirs(traces_folder, exist_ok=True)

for result in agent_results:
    image_name = result["image"]

    trace = {
        "input": image_name,
        "perception": {
            "yolo_objects": result["detected_objects"],
            "yolo_object_count": result["object_count"],
            "clip_scene": result["scene"],
            "clip_confidence": result["clip_confidence"]
        },
        "decision": {
            "result": result["decision"],
            "reason": result["reason"]
        },
        "action": {
            "annotated_image": f"results/images/annotated_{os.path.splitext(image_name)[0]}.jpg",
            "structured_results": "results/agent_results.json"
        }
    }

    trace_name = f"{os.path.splitext(image_name)[0]}_trace.json"
    trace_path = os.path.join(traces_folder, trace_name)

    with open(trace_path, "w") as f:
        json.dump(trace, f, indent=4)

    print(f"✓ Saved trace: {trace_path}")

print(f"\n✓ Trace logging complete for {len(agent_results)} images!")

✓ Saved trace: results/traces/sunset_trace.json
✓ Saved trace: results/traces/airport_trace.json
✓ Saved trace: results/traces/car lot_trace.json
✓ Saved trace: results/traces/dogs_trace.json
✓ Saved trace: results/traces/morocco market_trace.json
✓ Saved trace: results/traces/car_trace.json
✓ Saved trace: results/traces/downtown_trace.json
✓ Saved trace: results/traces/cars at night_trace.json
✓ Saved trace: results/traces/beautiful_trace.json
✓ Saved trace: results/traces/rain_trace.json

✓ Trace logging complete for 10 images!


In [32]:
# Cell 23 - Final evaluation metrics for all 10 test images

total_images = len(agent_results)

successful_runs = sum(
    1 for result in agent_results
    if result["decision"] is not None
)

review_cases = sum(
    1 for result in agent_results
    if "REVIEW" in result["decision"]
)

success_rate = (
    successful_runs / total_images * 100
    if total_images > 0 else 0
)

review_rate = (
    review_cases / total_images * 100
    if total_images > 0 else 0
)

average_yolo_time = (
    sum(result["yolo_inference_time"] for result in agent_results)
    / total_images
    if total_images > 0 else 0
)

print("FINAL AGENT EVALUATION")
print("-" * 45)
print(f"Total test images: {total_images}")
print(f"Successful end-to-end runs: {successful_runs}")
print(f"Pipeline completion rate: {success_rate:.1f}%")
print(f"Cases flagged for review: {review_cases}")
print(f"Review rate: {review_rate:.1f}%")
print(f"Average YOLO inference time: {average_yolo_time:.3f} seconds")
print("-" * 45)

# Save final metrics
metrics_text = f"""
AUTO SCENE AGENT - FINAL EVALUATION
-----------------------------------
Total test images: {total_images}
Successful end-to-end runs: {successful_runs}
Pipeline completion rate: {success_rate:.1f}%
Cases flagged for human review: {review_cases}
Review rate: {review_rate:.1f}%
Average YOLO inference time: {average_yolo_time:.3f} seconds

Note:
Pipeline completion measures whether the full agent completed
successfully. It does not mean every model prediction was correct.
"""

with open("results/metrics.txt", "w") as f:
    f.write(metrics_text)

print("\n✓ Final metrics saved to results/metrics.txt")

FINAL AGENT EVALUATION
---------------------------------------------
Total test images: 10
Successful end-to-end runs: 10
Pipeline completion rate: 100.0%
Cases flagged for review: 4
Review rate: 40.0%
Average YOLO inference time: 0.212 seconds
---------------------------------------------

✓ Final metrics saved to results/metrics.txt


In [33]:
# Cell 24 - Error handling and input validation

def validate_image(image_path):
    """
    Checks whether an input image exists and can be opened.
    Returns a structured status instead of crashing.
    """

    if not os.path.exists(image_path):
        return {
            "status": "error",
            "image": image_path,
            "message": "Image file was not found."
        }

    try:
        image = Image.open(image_path)
        image.verify()

        return {
            "status": "success",
            "image": image_path,
            "message": "Image is valid and ready for processing."
        }

    except Exception as error:
        return {
            "status": "error",
            "image": image_path,
            "message": f"Invalid image: {str(error)}"
        }


# Test one valid input
valid_test = validate_image(valid_images[0])

# Test one intentionally missing input
invalid_test = validate_image("missing_test_image.jpg")

print("VALID INPUT TEST")
print(json.dumps(valid_test, indent=4))

print("\nINVALID INPUT TEST")
print(json.dumps(invalid_test, indent=4))

print("\n✓ Error handling test complete!")

VALID INPUT TEST
{
    "status": "success",
    "image": "data/sample/sunset.jpg",
    "message": "Image is valid and ready for processing."
}

INVALID INPUT TEST
{
    "status": "error",
    "image": "missing_test_image.jpg",
    "message": "Image file was not found."
}

✓ Error handling test complete!


## Failure Analysis

The agent completed all 10 test cases without crashing, but the computer vision models were not always correct.

### Failure Case 1 – Cars at Night
YOLO detected two wine glasses instead of vehicles in the nighttime traffic image. CLIP correctly identified the image as a road or traffic scene with 81.6% confidence. Because the two models disagreed, the agent flagged the image for review instead of automatically accepting either prediction.

This shows that low lighting and unusual visual conditions can reduce object detection accuracy. It also shows why combining two vision tools can be useful.

### Failure Case 2 – Beautiful.jpg
CLIP classified the image as a dog scene with 77.0% confidence, while YOLO detected a bench and a potted plant and did not detect a dog. The agent recognized the disagreement and recommended review.

This shows that CLIP's scene classification depends heavily on the text labels provided to it. If the available labels do not represent the image well, CLIP may select the closest option even when it is incorrect.

### Additional Limitations
The downtown and rain images also showed uncertainty. YOLO detected no objects in either image, while CLIP had low confidence. The agent responded by recommending manual review instead of making a strong automatic decision.

These results show why human review is still important when computer vision systems are uncertain or when multiple models disagree.